# Car Price Prediction — Data Preprocessing

This notebook prepares the AutoScout24 car listings dataset for machine learning.

Based on the findings from the Exploratory Data Analysis (EDA), we will:

- Remove exact duplicate records.
- Handle missing and potentially unreliable values.
- Convert features into appropriate data types.
- Prepare numerical and categorical features for model training.
- Separate training and test data while avoiding data leakage.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
file_path = project_root / "data" / "cars" / "autoscout24_dataset_20251108.csv"

columns = ["make", "model", "price", "price_currency", "mileage_km", "registration_date", "production_year", "fuel_category", "transmission", "power_hp", "is_used", "country_code", "vehicle_type", "body_type"]

df = pd.read_csv(file_path, usecols=columns)

print("Original dataset:", df.shape)

Original dataset: (118382, 14)


## Duplicate Removal

The exploratory analysis identified 6,067 duplicate rows across the 14 selected columns.

Exact duplicates may overrepresent certain vehicle listings and negatively affect model evaluation.

We remove repeated rows while keeping the first occurrence of each unique record. Since listing identifiers are not available, identical records are treated as duplicates for modeling purposes.

In [3]:
df_clean = df.drop_duplicates().copy()

print("Original records:", len(df))
print("Removed duplicates:", len(df) - len(df_clean))
print("Remaining records:", len(df_clean))

Original records: 118382
Removed duplicates: 6067
Remaining records: 112315


## Price Cleaning

The EDA identified potentially unreliable vehicle prices, including listings priced at €1 and unusually expensive vehicles.

Since `price` is our target variable, incorrect values could negatively affect model training.

We will examine extreme prices before defining cleaning rules, avoiding the automatic removal of expensive vehicles that may have legitimate market values.

In [4]:
print("Cars priced at 1 €:", (df_clean["price"] == 1).sum())

print("Cars priced between 2 € and 999 €:", df_clean["price"].between(2, 999).sum())

Cars priced at 1 €: 8
Cars priced between 2 € and 999 €: 260


### Removing Placeholder Prices

We identified 8 vehicles listed at €1, which are likely placeholder prices rather than genuine selling prices.

We decided to remove these records while keeping other low-priced vehicles for further investigation.

In [9]:
before = len(df_clean)

df_clean = df_clean[df_clean["price"] != 1].copy()

print("Removed records:", before - len(df_clean))
print("Remaining records:", len(df_clean))

Removed records: 8
Remaining records: 112307


In [11]:
registration_year = pd.to_datetime(df_clean["registration_date"], errors="coerce").dt.year

suspicious_recent = df_clean[(registration_year > 2015) & (df_clean["price"] < 1000)]

print("Recent cars priced below 1,000 €:", len(suspicious_recent))

suspicious_recent[["make", "model", "registration_date", "mileage_km", "price", "is_used"]].sort_values("price").head(30)

Recent cars priced below 1,000 €: 1


,make,model,registration_date,mileage_km,price,is_used
64295,Suzuki,Celerio,2016-10-01,"240,000 km",699.0,True


### Removing Suspicious Recent Listings

We investigated vehicles registered after 2015 with prices below €1,000.

Three records were identified as potentially unreliable due to unusually low prices combined with suspicious vehicle characteristics.

We decided to exclude these records while retaining the Suzuki Celerio, whose age and high mileage make its low price more plausible.

These exclusions are based on a manual review and do not necessarily indicate errors in the original listings.

In [10]:
rows_to_remove = [38947, 71401, 57577]

print(df_clean.loc[rows_to_remove, ["make", "model", "price"]])

df_clean = df_clean.drop(index=rows_to_remove).copy()

print("Removed records:", len(rows_to_remove))
print("Remaining records:", len(df_clean))

             make     model  price
38947  Alfa Romeo    Tonale  450.0
71401       Volvo       V70  900.0
57577      Suzuki  Wagon R+  999.0
Removed records: 3
Remaining records: 112304


### Investigating Extremely High Prices

The EDA identified vehicles listed for more than €1,000,000.

While some prices may be legitimate for rare luxury vehicles, others may represent data entry errors.

We will examine the distribution of these listings across brands before deciding whether any records should be removed.

In [12]:
expensive_cars = df_clean[df_clean["price"] > 1000000].copy()

print("Cars over €1,000,000:", len(expensive_cars))

expensive_cars.groupby("make")["price"].agg(["count", "min", "median", "max"]).sort_values("count", ascending=False)

Cars over €1,000,000: 52


,count,min,median,max
make,,,,
Bugatti,23,1895000.0,5300000.0,13500000.0
Aston Martin,13,1310000.0,3450000.0,4641000.0
Porsche,8,1290990.0,1619975.0,3800000.0
Mercedes-Benz,4,1814750.0,3249975.0,3750000.0
Maybach,2,1279000.0,1496300.0,1713600.0
BMW,1,9500000.0,9500000.0,9500000.0
Rolls-Royce,1,1011500.0,1011500.0,1011500.0


In [14]:
df_clean.loc[df_clean["make"].isin(["BMW", "Mercedes-Benz"]) & (df_clean["price"] > 1000000), ["make", "model", "registration_date", "mileage_km", "power_hp", "price"]].sort_values("price", ascending=False)

,make,model,registration_date,mileage_km,power_hp,price
16344,BMW,316,2015-03-01,"200,000 km",116.0,9500000.0
2508,Mercedes-Benz,AMG ONE,2024-10-01,50 km,1063.0,3750000.0
4373,Mercedes-Benz,AMG ONE,2025-03-01,10 km,1063.0,3629950.0
3789,Mercedes-Benz,AMG ONE,2024-10-01,1 km,1445.0,2870000.0
112002,Mercedes-Benz,G 63 AMG,2025-10-01,93 km,799.0,1814750.0


### Removing an Implausible High-Price Listing

We identified a BMW 316 from 2015, with 116 hp and 200,000 km, listed for €9,500,000.

This price is extremely inconsistent with the vehicle's characteristics, so we decided to exclude this record as a likely pricing error.

Other high-priced vehicles were retained, since their prices may be legitimate for luxury or limited-production models, in this  case the Mercedes taht appear are rare and very expensive cars.

In [15]:
invalid_bmw = (df_clean["make"] == "BMW") & (df_clean["model"] == "316") & (df_clean["price"] == 9500000)

print("Records to remove:", invalid_bmw.sum())

df_clean = df_clean.loc[~invalid_bmw].copy()

print("Remaining records:", len(df_clean))

Records to remove: 1
Remaining records: 112303


## Mileage Cleaning

The EDA identified two issues with the `mileage_km` column:

- Mileage values are stored as text, such as `"150,000 km"`.
- Some vehicles have exceptionally high mileage values that may indicate data entry errors.

We will convert mileage into a numeric format and investigate extreme values before deciding whether any records should be excluded.

In [16]:
df_clean["mileage_numeric"] = pd.to_numeric(df_clean["mileage_km"].str.replace(" km", "", regex=False).str.replace(",", "", regex=False), errors="coerce")

print("Missing mileage:", df_clean["mileage_numeric"].isna().sum())
print("Failed conversions:", (df_clean["mileage_km"].notna() & df_clean["mileage_numeric"].isna()).sum())

Missing mileage: 272
Failed conversions: 0


In [17]:
print("Cars over 500,000 km:", (df_clean["mileage_numeric"] > 500000).sum())

print("Cars over 1,000,000 km:", (df_clean["mileage_numeric"] > 1000000).sum())

df_clean.nlargest(15, "mileage_numeric")[["make", "model", "registration_date", "mileage_numeric", "price", "is_used"]]

Cars over 500,000 km: 20
Cars over 1,000,000 km: 3


,make,model,registration_date,mileage_numeric,price,is_used
15651,BMW,320,2002-09-01,2570000.0,5500.0,True
15254,BYD,Atto 3,2023-08-01,2424696.0,26990.0,True
7354,Volvo,C30,2007-07-01,1560000.0,2000.0,True
64278,BMW,318,2003-01-01,1000000.0,500.0,True
77399,BMW,X4 M,2017-03-01,900000.0,47500.0,True
29550,Volvo,S60,2001-10-01,858890.0,1650.0,True
38452,Volvo,V70,2002-07-01,740315.0,2200.0,True
108213,BMW,X5,2012-08-01,637525.0,9990.0,True
69704,Volvo,XC70,2005-05-01,626349.0,2443.0,True
78921,BMW,740,2001-04-01,600000.0,4995.0,True


### Removing Suspicious Mileage Records

We identified four vehicles with extremely unusual mileage values, ranging from 900,000 km to over 2.5 million km.

A BYD from 2023 is impossible to have so many km and that price, BMW X4 also impossible to have so many km costing 47500 and the other two extremely high km traveled

These records were excluded as a conservative data quality measure because their mileage could not be considered sufficiently reliable for model training.

Other high-mileage vehicles were retained, since high mileage alone does not necessarily indicate incorrect data.

In [19]:
suspicious_mileage = [15651, 15254, 7354, 77399]

print(df_clean.loc[suspicious_mileage, ["make", "model", "mileage_numeric", "price"]])

df_clean = df_clean.drop(index=suspicious_mileage).copy()

print("Remaining records:", len(df_clean))

        make   model  mileage_numeric    price
15651    BMW     320        2570000.0   5500.0
15254    BYD  Atto 3        2424696.0  26990.0
7354   Volvo     C30        1560000.0   2000.0
77399    BMW    X4 M         900000.0  47500.0
Remaining records: 112299


## Vehicle Age Preparation

Vehicle age is an important factor in car price prediction.

During EDA, we found that `production_year` contains a large number of missing values. However, the registration year closely matches the production year for most vehicles where both values are available.

We will extract the registration year from `registration_date` and check for missing or potentially invalid values before using it as a model feature.

### Extracting the Registration Year

We convert `registration_date` into a date format and extract the year into a new column called `registration_year`.

We will then check for missing values and unusual registration years to ensure the data is suitable for our prediction model.

In [21]:
df_clean["registration_year"] = pd.to_datetime(df_clean["registration_date"], errors="coerce").dt.year

print("Missing registration years:", df_clean["registration_year"].isna().sum())

print("Oldest registration year:", df_clean["registration_year"].min())

print("Newest registration year:", df_clean["registration_year"].max())

df_clean.groupby("is_used")["registration_year"].agg(["count", "min", "median", "max"])

Missing registration years: 203
Oldest registration year: 1929.0
Newest registration year: 2026.0


,count,min,median,max
is_used,,,,
False,19119,1929.0,2025.0,2026.0
True,92977,1934.0,2020.0,2026.0


### Registration Year Validation

The registration years range from 1929 to 2026, with 203 missing values.

We will investigate vehicles registered after the dataset's 2025 reference year and unusually old vehicles classified as new.

These checks will help us identify potential inconsistencies before creating a vehicle age feature.

In [22]:
print("Registrations after 2025:", (df_clean["registration_year"] > 2025).sum())

print("New vehicles registered before 1950:", ((df_clean["is_used"] == False) & (df_clean["registration_year"] < 1950)).sum())

df_clean.loc[df_clean["registration_year"] > 2025, ["make", "model", "registration_date", "production_year", "is_used", "price"]].head(10)

Registrations after 2025: 110
New vehicles registered before 1950: 4


,make,model,registration_date,production_year,is_used,price
1274,BMW,iX3,2026-01-01,NaN,False,90096.0
1359,Porsche,992,2026-01-01,NaN,False,372221.0
1761,Audi,Q2,2026-01-01,NaN,False,53090.0
1786,Volvo,XC90,2026-02-01,NaN,True,78945.0
1791,Audi,A6,2026-01-01,NaN,False,84950.0
1984,Audi,Q5,2026-01-01,NaN,True,36880.0
2516,Audi,Q4 e-tron,2026-01-01,NaN,False,54989.0
2537,Audi,Q3,2026-01-01,NaN,False,57990.0
2724,Audi,Q5,2026-05-01,NaN,False,75990.0
2871,Audi,A3,2026-01-01,NaN,False,49989.0


In [23]:
df_clean.loc[df_clean["registration_year"] == 2026, "is_used"].value_counts()

is_used
False    97
True     13
Name: count, dtype: int64

In [24]:
df_clean.loc[(df_clean["is_used"] == False) & (df_clean["registration_year"] < 1950), ["make", "model", "registration_date", "production_year", "mileage_numeric", "price"]]

,make,model,registration_date,production_year,mileage_numeric,price
1902,Rolls-Royce,Phantom,1929-01-01,NaN,10236.0,198950.0
33068,Bugatti,NaN,1935-01-01,NaN,5000.0,599000.0
79133,Rolls-Royce,NaN,1933-01-01,NaN,66386.0,298000.0
79643,Rolls-Royce,NaN,1934-01-01,NaN,48000.0,130000.0


### Removing Inconsistent Vehicle Records

The registration year validation identified two potential data quality issues:

- **110 vehicles** with registration years after the dataset's 2025 reference year.
- **4 classic vehicles** registered before 1950 but classified as new, despite having substantial mileage.

We decided to exclude these records as a conservative data cleaning measure.

This decision does not necessarily mean the original listings are incorrect, but their characteristics are inconsistent with the assumptions used in our prediction model.

In [27]:
future_cars = df_clean["registration_year"] > 2025

incorrect_new_cars = (df_clean["is_used"] == False) & (df_clean["registration_year"] < 1950)

print("Future registrations:", future_cars.sum())
print("Inconsistent new cars:", incorrect_new_cars.sum())

before = len(df_clean)

rows_to_remove = df_clean[future_cars | incorrect_new_cars].index

df_clean = df_clean.drop(index=rows_to_remove).copy()

print("Removed records:", before - len(df_clean))
print("Remaining records:", len(df_clean))

Future registrations: 0
Inconsistent new cars: 0
Removed records: 0
Remaining records: 112185


### Creating the Vehicle Age Feature

We calculate vehicle age using 2025 as the dataset's reference year.

The new `vehicle_age` feature represents the number of years since the vehicle's first registration.

This feature may help the model learn how vehicle age is associated with market price.

Records with missing registration years will be preserved and handled later during model preprocessing.

In [28]:
reference_year = 2025

df_clean["vehicle_age"] = reference_year - df_clean["registration_year"]

print("Missing vehicle ages:", df_clean["vehicle_age"].isna().sum())

print("Minimum age:", df_clean["vehicle_age"].min())
print("Maximum age:", df_clean["vehicle_age"].max())

df_clean.groupby("is_used")["vehicle_age"].agg(["count", "min", "median", "max"]).round(2)

Missing vehicle ages: 203
Minimum age: 0.0
Maximum age: 91.0


,count,min,median,max
is_used,,,,
False,19018,0.0,0.0,70.0
True,92964,0.0,5.0,91.0


### Checking Vehicle Age and Condition Consistency

The vehicle age feature contains 203 missing values, with ages ranging from 0 to 91 years.

The median age is 0 years for new vehicles and 5 years for used vehicles. However, some vehicles classified as new are considerably older.

We will inspect vehicles classified as new with more than 10 years of age to identify potential inconsistencies in the `is_used` feature.

In [29]:
old_new_cars = (df_clean["is_used"] == False) & (df_clean["vehicle_age"] > 10)

print("New vehicles older than 10 years:", old_new_cars.sum())

df_clean.loc[old_new_cars, ["make", "model", "registration_year", "vehicle_age", "mileage_numeric", "price"]].sort_values("vehicle_age", ascending=False).head(15)

New vehicles older than 10 years: 39


,make,model,registration_year,vehicle_age,mileage_numeric,price
77610,Rolls-Royce,Silver Dawn,1955.0,70.0,1234.0,65500.0
104850,Aston Martin,DB,1958.0,67.0,77000.0,199000.0
57479,Rolls-Royce,Cloud,1960.0,65.0,91756.0,89500.0
29724,Rolls-Royce,Cloud,1960.0,65.0,66795.0,55000.0
20215,Rolls-Royce,Cloud,1965.0,60.0,60029.0,39999.0
27198,Rolls-Royce,Cloud,1965.0,60.0,34084.0,98500.0
95671,Rolls-Royce,Cloud,1965.0,60.0,112506.0,49999.0
46056,Rolls-Royce,Silver Shadow,1967.0,58.0,55984.0,52500.0
23107,Maserati,Indy,1970.0,55.0,72772.0,69900.0
30896,Rolls-Royce,Silver Shadow,1971.0,54.0,3000.0,24000.0


### Removing Missing Vehicle Ages

We identified 203 vehicles with missing registration years, representing approximately 0.18% of the dataset.

Since vehicle age is an important feature for price prediction and the number of affected records is very small, we decided to remove these vehicles rather than estimate their ages.

This simplifies preprocessing while preserving more than 99% of the available records.

In [30]:
before = len(df_clean)

df_clean = df_clean.dropna(subset=["vehicle_age"]).copy()

print("Removed records:", before - len(df_clean))
print("Remaining records:", len(df_clean))
print("Missing vehicle ages:", df_clean["vehicle_age"].isna().sum())

Removed records: 203
Remaining records: 111982
Missing vehicle ages: 0


### Investigating Mileage in Vehicles Classified as New

We will investigate all vehicles classified as new (`is_used = False`) with more than 2,000 km, regardless of their age.

The 2,000 km threshold is used to identify potentially inconsistent condition labels, not as a strict definition of a new vehicle.

We will examine the number of affected records and their mileage distribution before deciding whether any additional cleaning is necessary.

In [32]:
new_high_mileage = (df_clean["is_used"] == False) & (df_clean["mileage_numeric"] > 2000)

print("Total new vehicles:", (df_clean["is_used"] == False).sum())
print("New vehicles above 2,000 km:", new_high_mileage.sum())

df_clean.loc[new_high_mileage, "mileage_numeric"].describe()

df_clean.loc[new_high_mileage, ["make", "model", "vehicle_age", "mileage_numeric", "price"]].sort_values("mileage_numeric", ascending=False).head(20)

Total new vehicles: 19018
New vehicles above 2,000 km: 12375


,make,model,vehicle_age,mileage_numeric,price
98830,Alfa Romeo,Giulia,9.0,204700.0,7999.0
115168,Porsche,911,18.0,198713.0,49999.0
107012,Rolls-Royce,Silver Shadow,52.0,186365.0,7500.0
3909,Porsche,Cayenne,14.0,185000.0,18999.0
14014,BMW,318,5.0,182182.0,18990.0
98033,Rolls-Royce,Silver Shadow,48.0,179901.0,16950.0
43167,BMW,M5,7.0,174490.0,56999.0
112320,Ford,Transit Custom,11.0,172000.0,9999.0
14631,Rolls-Royce,Silver Spirit,37.0,170420.0,26900.0
74426,BMW,520,5.0,161795.0,25490.0


### Decision: Excluding Vehicle Condition

The analysis revealed potential inconsistencies in the `is_used` feature. Of the 19,018 vehicles classified as new, 12,375 have more than 2,000 km, and some have exceptionally high mileage.

Although vehicle condition can influence market prices, its reliability in this dataset is uncertain.

We decided to exclude `is_used` from the initial model and rely on `vehicle_age` and `mileage_numeric` to represent vehicle age and usage.

The affected vehicle records will be retained, as their other characteristics may still be useful for price prediction.

### Exploring Additional Categorical Features

Before finalizing feature selection, we will inspect four categorical variables: `transmission`, `body_type`, `vehicle_type`, and `country_code`.

We want to understand which categories are present, how frequently they occur, and whether these features provide useful information for price prediction.

This will help us decide which variables should be included in the initial model.

In [35]:
print("Transmission:")
print(df_clean["transmission"].value_counts(dropna=False).head(10))

print("\nBody type:")
print(df_clean["body_type"].value_counts(dropna=False).head(10))

print("\nVehicle type:")
print(df_clean["vehicle_type"].value_counts(dropna=False).head(10))

print("\nCountry code:")
print(df_clean["country_code"].value_counts(dropna=False).head(10))

Transmission:
transmission
Automatic         90803
Manual            16460
Semi-automatic     3916
NaN                 803
Name: count, dtype: int64

Body type:
body_type
Off-Road/Pick-up    42396
Sedan               23652
Station wagon       15584
Coupe               13688
Convertible          7355
Compact              4435
Van                  2627
Other                1159
Transporter           722
Panel van             184
Name: count, dtype: int64

Vehicle type:
vehicle_type
Car            111574
Transporter       408
Name: count, dtype: int64

Country code:
country_code
DE     43845
IT     22870
NL     15753
BE      9187
AT      6971
ES      6468
FR      6091
LU       782
NaN       15
Name: count, dtype: int64


### Additional Categorical Features — Findings

The analysis showed that `transmission`, `body_type`, and `country_code` contain potentially useful information for price prediction.

However, `vehicle_type` is highly imbalanced, with approximately 99.6% of vehicles classified as cars. Since `body_type` already provides more detailed vehicle categories, we decided to exclude `vehicle_type` from the initial model.

We will retain `transmission`, `body_type`, and `country_code`. Missing values will be handled during model preprocessing.

## Feature Selection

We selected nine features for our initial car price prediction model: three numerical and six categorical.

We excluded redundant, constant, highly incomplete, or potentially unreliable features, including `production_year`, `price_currency`, `vehicle_type`, and `is_used`.

The target variable is `price`, while the selected features will be used as model inputs.

In [36]:
numerical_features = ["mileage_numeric", "vehicle_age", "power_hp"]

categorical_features = ["make", "model", "fuel_category", "transmission", "body_type", "country_code"]

features = numerical_features + categorical_features

X = df_clean[features].copy()
y = df_clean["price"].copy()

print("Number of records:", len(X))
print("Number of features:", X.shape[1])

Number of records: 111982
Number of features: 9


### Checking Missing Values in Selected Features

Before splitting the dataset, we will check the selected features for missing values.

This will help us determine which preprocessing techniques are required to prepare the data for model training.

In [37]:
print(X.isna().sum().sort_values(ascending=False))

power_hp           1462
model              1276
transmission        803
mileage_numeric     234
fuel_category        71
country_code         15
vehicle_age           0
make                  0
body_type             0
dtype: int64


### Handling Missing Values in Selected Features

The selected features contain relatively few missing values, with each column having less than 2% missing data.

Since our dataset contains approximately 112,000 vehicles, we decided to remove records with missing values in any of the selected features rather than estimate unknown values.

This simplifies the initial modeling process while preserving most of the dataset.

We will keep only complete records for model training and evaluation.

In [38]:
before = len(df_clean)

df_clean = df_clean.dropna(subset=features).copy()

X = df_clean[features].copy()
y = df_clean["price"].copy()

print("Removed records:", before - len(df_clean))
print("Remaining records:", len(df_clean))
print("Remaining missing values:", X.isna().sum().sum())

Removed records: 3709
Remaining records: 108273
Remaining missing values: 0


**Findings:**

We removed 3,709 records containing missing values in the selected features, representing approximately 3.31% of the dataset at this stage.

The remaining dataset contains 108,273 vehicles and no missing values in the nine selected features.

The data is now ready to be divided into training and testing sets.

## Train-Test Split

We split the cleaned dataset into training (80%) and testing (20%) sets.

The training set will be used to teach the model the relationship between vehicle characteristics and price.

The testing set will be kept separate to evaluate the model's performance on unseen data.

We use a fixed random state to ensure reproducible results.

In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training records: 86618
Testing records: 21655
Training features: (86618, 9)
Testing features: (21655, 9)


## Categorical Encoding

Machine learning algorithms generally require numerical inputs, but our dataset contains six categorical features.

We will use One-Hot Encoding to transform categorical values into numerical representations without introducing an artificial order between categories.

Numerical features will remain unchanged.

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(transformers=[
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numerical", "passthrough", numerical_features)
])

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Training shape:", X_train_encoded.shape)
print("Testing shape:", X_test_encoded.shape)

Training shape: (86618, 662)
Testing shape: (21655, 662)


**Findings:**

One-Hot Encoding expanded the original 9 features into 662 numerical features, while preserving all 86,618 training records and 21,655 testing records.

Both datasets now have the same feature structure and are ready for model training.